# Lesson 13 — Exercise Solutions

Worked solutions with a short rationale after each. Run setup.sql first. Use SSMS (Ctrl+M) to see the plan effects each rationale describes.

In [ ]:
USE AdventureWorks2022;
GO
SET STATISTICS IO   ON;
SET STATISTICS TIME ON;

## Exercise 1 — SARGable rewrite

Run the query below and look at its execution plan. Identify: (a) the plan operator on Sales.SalesOrderDetail, (b) estimated vs actual rows, (c) logical reads. Then rewrite the WHERE clause to be SARGable and compare.

In [ ]:
SELECT soh.SalesOrderID, SUM(sod.LineTotal) AS OrderTotal
FROM lesson13.SalesOrderHeader AS soh
JOIN Sales.SalesOrderDetail    AS sod ON sod.SalesOrderID = soh.SalesOrderID
WHERE soh.OrderDate >= '2013-07-01' AND soh.OrderDate < '2013-08-01'  -- SARGable
GROUP BY soh.SalesOrderID;

**Why:** CONVERT() on the column is non-SARGable and forces a scan; a date range lets the index on OrderDate seek. Expected: fewer logical reads and an Index Seek instead of a Scan on SalesOrderHeader.

## Exercise 2 — eliminate a key lookup

The query below triggers a key lookup. Add an index to lesson13.SalesOrderHeader to eliminate it, then verify with the execution plan that the key lookup is gone.

In [ ]:
DROP INDEX IF EXISTS IX_lesson13_SOH_TerritoryID_Inc ON lesson13.SalesOrderHeader;  -- re-run guard
CREATE INDEX IX_lesson13_SOH_TerritoryID_Inc
    ON lesson13.SalesOrderHeader (TerritoryID)
    INCLUDE (SalesPersonID, SubTotal);

SELECT SalesOrderID, SalesPersonID, SubTotal
FROM lesson13.SalesOrderHeader
WHERE TerritoryID = 4;

**Why:** keying on TerritoryID with INCLUDE (SalesPersonID, SubTotal) makes the index covering, so the Key Lookup operator disappears from the plan. The DROP INDEX IF EXISTS line is a re-run guard.

## Exercise 3 — selectivity and access path

Run both queries and compare their plans and logical reads. Explain why the plans differ.

In [ ]:
SELECT SalesOrderID, TotalDue FROM lesson13.SalesOrderHeader WHERE Status = 5;
SELECT SalesOrderID, TotalDue FROM lesson13.SalesOrderHeader WHERE Status = 1;

**Why:** Status = 5 (shipped) covers most rows — the optimizer may choose a Clustered Index Scan because seeking plus looking up each row would cost more than one full scan. Status = 1 (in process) covers very few rows, so it chooses an Index Seek. Selectivity drives the access-path choice.

## Exercise 4 — parameter-sniffing-safe procedure

Create a stored procedure lesson13.usp_RevenueByTerritory that accepts @TerritoryID INT and returns SUM(TotalDue) grouped by YEAR(OrderDate). Call it with territory 1 (large) and territory 10 (small) and check whether the plan is appropriate for both. Add OPTION (OPTIMIZE FOR (@TerritoryID UNKNOWN)) if needed.

In [ ]:
CREATE OR ALTER PROCEDURE lesson13.usp_RevenueByTerritory
    @TerritoryID INT
AS
    SELECT
        YEAR(OrderDate)  AS OrderYear,
        SUM(TotalDue)    AS Revenue
    FROM lesson13.SalesOrderHeader
    WHERE TerritoryID = @TerritoryID
    GROUP BY YEAR(OrderDate)
    ORDER BY OrderYear
    OPTION (OPTIMIZE FOR (@TerritoryID UNKNOWN));

In [ ]:
EXEC lesson13.usp_RevenueByTerritory @TerritoryID = 1;
EXEC lesson13.usp_RevenueByTerritory @TerritoryID = 10;

**Why:** OPTIMIZE FOR UNKNOWN compiles the plan from average statistics instead of the first sniffed @TerritoryID, so one cached plan is reasonable for both the large territory (1) and the small one (10).

## Exercise 5 — JOIN vs correlated subquery

Use the SET STATISTICS IO output to compare logical reads for the two queries below. Which one is more efficient, and why?

In [ ]:
SELECT p.Name, SUM(sod.OrderQty) AS TotalSold
FROM Sales.SalesOrderDetail AS sod
JOIN Production.Product     AS p  ON p.ProductID = sod.ProductID
GROUP BY p.Name
ORDER BY TotalSold DESC;

In [ ]:
SELECT p.Name,
       (SELECT SUM(OrderQty) FROM Sales.SalesOrderDetail sod WHERE sod.ProductID = p.ProductID) AS TotalSold
FROM Production.Product AS p
ORDER BY TotalSold DESC;

**Why:** the JOIN makes one pass over SalesOrderDetail with a Hash Match aggregate. The correlated subquery aggregates SalesOrderDetail once per product — ~500 executions for ~500 products — so Query A has far fewer logical reads. Correlated aggregates over a large table are almost always better written as a JOIN with GROUP BY.

In [ ]:
SET STATISTICS IO   OFF;
SET STATISTICS TIME OFF;